$$
\text{Sign}(x) \;=\; \text{Sign}_{0,1,0}(x) \;=\; \begin{cases} 1 & x > 0 \\ 0 & x = 0 \\ -1 & x < 0 \end{cases}
$$


$$
\text{Sign}_{a,b,c}(x) \;=\; a + (b-a)\,\text{Sign}(x - c) \;=\; \begin{cases} b & x > c \\ \dfrac{a+b}{2} & x = c \\ a & x < c \end{cases}
$$

$$
\begin{aligned}
(x == a) &:= (x > a - \epsilon) \cdot (x < a + \epsilon) \\
(x_i == 0) &:= (x_i \geq -0.01) \cdot (x_i \leq 0.01) \\[6pt]
x_i = 0.005: \quad &(0.005 \geq -0.01) \cdot (0.005 \leq 0.01) = 1 \cdot 1 = 1 \\
x_i = 0.5: \quad &(0.5 \geq -0.01) \cdot (0.5 \leq 0.01) = 1 \cdot 0 = 0
\end{aligned}
$$

## Tanh Sign Convergence 

### Plotted Approxiation Dataset Output

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Load data (long format: one row per (k, x); x in [-2, 2], 0.001 step)
df = pd.read_csv("tests/sign_tanh_convergence_results.csv")

# The 9-term Taylor series for tanh only converges for |k*x| < pi/2, so each
# approximation diverges (up to ~1e28 for k=64) once x leaves |x| < pi/(2k).
# Mask the divergent tails to NaN so each line draws only over its valid
# region; Plotly breaks the line at NaNs, leaving clean central curves.
MASK_THRESHOLD = 2.0
df["tanh_approximation"] = df["tanh_approximation"].where(
    df["tanh_approximation"].abs() <= MASK_THRESHOLD, np.nan
)

# Treat k as a discrete category so each k gets its own colored line/legend.
df["k"] = df["k"].astype(str)


def plot_tanh_sign(df, k_list, title, x_range=(-2, 2)):
    """One masked approximation line per k in k_list, plus the True Sign step."""
    order = [str(k) for k in k_list]
    sub = df[df["k"].isin(order)]

    fig = px.line(
        sub,
        x="x_value",
        y="tanh_approximation",
        color="k",
        category_orders={"k": order},
        title=title,
        labels={
            "x_value": "Plaintext Input",
            "tanh_approximation": "Calculated Value",
            "k": "k",
        },
    )

    # Overlay the true sign function once (identical across k) as a reference.
    true = sub[sub["k"] == order[0]]
    fig.add_trace(
        go.Scatter(
            x=true["x_value"],
            y=true["sign_real"],
            mode="lines",
            name="True Sign",
            line=dict(color="black", dash="dash", width=2),
        )
    )

    fig.update_xaxes(range=list(x_range))
    fig.update_yaxes(range=[-0.5, 1.5])
    return fig


# Graph 1: low k -- wide valid windows, so show the full x-range.
plot_tanh_sign(
    df,
    [1, 2, 4],
    "Tanh Sign Approximation (n_terms = 9) vs True Sign — low k (1, 2, 4)",
    x_range=(-2, 2),
).show()

In [ ]:
# Graph 2: high k -- narrow valid windows (k=64 is only +/-0.03), so zoom in
# on x near 0 to make the steep transitions readable.
plot_tanh_sign(
    df,
    [8, 16, 32, 64],
    "Tanh Sign Approximation (n_terms = 9) vs True Sign — high k (8, 16, 32, 64)",
    x_range=(-0.3, 0.3),
).show()